# Wavelet Scattering Transform Examples

This notebook demonstrates how to use the Wavelet Scattering Transform (WST) estimator from the ACM package.

The WST is a powerful tool for extracting non-Gaussian information from galaxy density fields through a combination of wavelet decomposition and statistical moments.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from helpers import load_estimator_parameters, make_lagrangian_mock

from acm import setup_logging
from acm.estimators.galaxy_clustering.backends.jaxpower import (
    JaxpowerBackend,  # noqa: F401 - register backend
)
from acm.estimators.galaxy_clustering.wst import WaveletScatteringTransform

setup_logging()

In [ ]:
los = "z"
data_positions, boxsize = make_lagrangian_mock(boxsize=500.0, los=los)

# Instanciate class
estimator = WaveletScatteringTransform(
    backend='jaxpower',
    data_positions=data_positions,
    J=4,
    L=4,
    sigma=0.8,
    integral_powers=(0.8,), # Only accepts 1 integral power for now
    frontend='torch',
    boxsize=boxsize, # Backend initialization arguments
    cellsize=100.0, # Using a larger cellsize (e.g., 100 Mpc/h) for the WST computation
)
# Set the density contrast field
estimator.backend.set_density_contrast()

# Compute WST coefficients
result = estimator.compute()

print(f"Number of WST coefficients: {len(result.coefficients)}")
print(f"First 10 coefficients: {result.coefficients[:10]}")

## Visualizing WST Coefficients

Let's plot the scattering coefficients to see the distribution of wavelet features.

In [ ]:
fig, ax = estimator.plot(result, marker="o", ls='-', markersize=3)
ax.set_title('Wavelet Scattering Transform Coefficients')
ax.grid(True, alpha=0.3)

## Pre-loading the kymatio object

As `kymatio` - the package used to compute the WST coefficients - can take some time to initialize, we can pre-compute save the initial object as a pickle. Here, we just precompute it and pas it directly.

In [ ]:
kymatio_object = WaveletScatteringTransform.initialize_kymatio(
    J=4,
    L=4,
    sigma_0=0.8,
    shape=(64, 64, 64), # This should match the backend meshsize !
    integral_powers=(0.8,),
    frontend='torch',
)

# import pickle
# from pathlib import Path

# with Path("kymatio_object.pkl").open("wb") as f:
#     pickle.dump(kymatio_object, f)

# with Path("kymatio_object.pkl").open("rb") as f:
#     kymatio_object = pickle.load(f)

estimator = WaveletScatteringTransform(
    backend='jaxpower',
    data_positions=data_positions,
    kymatio_object=kymatio_object, # Pass the pre-initialized kymatio object - all other parameters are ignored
    boxsize=boxsize, # Backend initialization arguments
    cellsize=100.0, # Using a larger cellsize (e.g., 100 Mpc/h) for the WST computation
)